In [1]:
# Please install OpenAI SDK first: `pip3 install openai`

from openai import OpenAI
from tqdm import tqdm
import threading



In [2]:
import os
from openai import OpenAI

client = OpenAI(
    api_key="6df4cc08-4f23-49b9-85f3-a9eb79084143",
    base_url = "https://ark.cn-beijing.volces.com/api/v3",
)



In [3]:
import json

# 方法一：使用 with 语句自动管理文件的打开和关闭
file_path = '../dataset/raw/test1.json'
try:
    with open(file_path, 'r', encoding='utf-8') as file:
        # 读取并解析 JSON 数据
        data = json.load(file)
        # 打印读取的数据
        print(data[0])
except FileNotFoundError:
    print(f"文件 {file_path} 未找到。")
except json.JSONDecodeError:
    print(f"文件 {file_path} 不是有效的 JSON 格式。")

{'id': 4169, 'content': '飞周兄弟还不认亲？兄弟啊，还袭击？'}


In [5]:
run_num = 100
mode = 'chat'

In [6]:



# 定义请求函数
def make_api_call(res_list, pos,one_num):
    for d in tqdm(data[pos:pos +one_num]):
        try:
            response = client.chat.completions.create(
    model = "doubao-1-5-pro-32k-250115",  # your model endpoint ID
    messages = [
        {"role": "system", "content": """
我将给你一个中文社交媒体文本作为输入，你需要识别并输出仇恨言论四元组，要求如下：
1.识别评论对象(Target)，无目标时写NULL；提取对应论点(Argument)；确定目标群体(Targeted Group)：如果是仇恨，判断是一下哪个：LGBTQ/Racism/Region/Sexism/others；如果不涉及仇恨就是non-hate；判断是否仇恨(Hateful)：hate/non-hate
2.多个四元组用[SEP]分隔，最后用[END]结尾,输出格式：Target | Argument | Targeted Group | Hateful [SEP]... [END]    
    
下面提供几个输入输出样例：

输入： 你可真是头蠢驴，这都做不好。
输出： 你 | 蠢驴 | non-hate | non-hate [END]

输入： 老黑我是真的讨厌，媚黑的还倒贴。
输出： 老黑 | 讨厌 | Racism | hate [SEP] 媚黑的 | 倒贴 | Racism | hate [END]

输入： 就最近来说有两个事，lol某主播的前女友发文称其飞黄腾达后抛弃她。我个人的想法是看男方的回应，本来恋爱期间有些东西是说不清的。结果小作文来了，一群人不等男方回应直接定论。如果男方承认了那确实应该锤死，万一有什么反转呢？这次是顺风团，万一下次是逆风呢？
输出： 一群人 | 不等男方回应直接定论 | others | hate [END]

输入： 图片暂时发不出来等我想想办法
输出： 图片 | 暂时发不出来 | non-hate | non-hate [END]
        """},
        {"role": "user", "content":  '请你逐步思考判断一下这个文本，然后在最后一行输出一个仇恨言论四元组：' + d["content"]},
    ],
)
            res_str = response.choices[0].message.content
            print(d["content"])
            print(res_str)
            res_list.append(res_str)
        except Exception as e:
            print(f"Error processing {d['content']}: {e}")

# 创建 200 个线程
final_res = []
threads = []
res_lists = []


one_num = int(len(data)/run_num)

for i in range(run_num):
    res = []
    res_lists.append(res)
    pos = i *one_num
    thread = threading.Thread(target=make_api_call, args=(res, pos,one_num))
    threads.append(thread)
    thread.start()

# 等待所有线程完成
for thread in threads:
    thread.join()

# 按顺序合并结果
for res in res_lists:
    final_res.extend(res)


  0%|                                                                                                                                                                                                             | 0/20 [00:00<?, ?it/s]


                                                                                                                                                                                                         | 0/20 [00:00<?, ?it/s]






                                                                                                                                                                                             | 0/20 [00:00<?, ?it/s]




                                                                                                                                                                                                   | 0/20 [00:00<?, ?it/s]

                                                                                               

反向操作去喜欢中国的地方留学不就行了
1. 首先分析评论对象（Target）：
   - 文本中没有明确指向某一个具体的人或事物作为评论对象，这里写NULL。
2. 接着提取对应论点（Argument）：
   - 论点是“反向操作去喜欢中国的地方留学”。
3. 然后确定目标群体（Targeted Group）：
   - 文本中没有体现出对特定群体的仇恨，属于无仇恨相关内容，所以是non - hate。
4. 最后判断是否仇恨（Hateful）：
   - 同样，文本不涉及仇恨内容，所以是non - hate。

NULL | 反向操作去喜欢中国的地方留学 | non - hate | non - hate [END]
我完全同意你找
1. 首先分析评论对象（Target）：从文本“我完全同意你找”中，可以看出“你”是被提到的对象，所以评论对象为“你”。
2. 接着看对应论点（Argument）：“完全同意”表明态度是赞同，所以论点是“完全同意”。
3. 然后确定目标群体（Targeted Group）：文本只是一种表达同意的正常话语，不涉及仇恨相关的特定群体，所以目标群体是“non - hate”。
4. 最后判断是否仇恨（Hateful）：由于只是表达同意，不带有仇恨性质，所以判断为“non - hate”。

你 | 完全同意 | non - hate | non - hate [END]
卡肺囊虫肺炎和隐脑大几率会死人的。
1. 首先分析评论对象（Target）：
   - 文本中主要提到了“卡肺囊虫肺炎”和“隐脑”这两种疾病，所以评论对象就是这两种疾病。
2. 接着看对应论点（Argument）：
   - 论点是“大几率会死人的”，这是对这两种疾病特点的描述。
3. 然后确定目标群体（Targeted Group）：
   - 这里的目标是疾病，不是针对特定的人群，所以目标群体属于non - hate。
4. 最后判断是否仇恨（Hateful）：
   - 文本只是在陈述疾病的危险性，没有仇恨的情感，所以是non - hate。

卡肺囊虫肺炎、隐脑 | 大几率会死人的 | non - hate | non - hate [END]
笑死了，eg的名头是老外安上去，关中国人什么事
1. 首先分析评论对象（Target）：
   - 从文本中可以看出，明

千万不要找大龄，尤其是从大城市回来的大龄剩女
1. 首先确定评论对象（Target）：
   - 从文本“千万不要找大龄，尤其是从大城市回来的大龄剩女”可以看出，评论对象是“从大城市回来的大龄剩女”。
2. 接着提取对应论点（Argument）：
   - 文本中表达的观点是“千万不要找”，所以论点是“不要找”。
3. 然后确定目标群体（Targeted Group）：
   - 这里针对的是特定的女性群体（大龄剩女），这种带有偏见性的言论属于性别歧视范畴，所以目标群体是“Sexism”。
4. 最后判断是否仇恨（Hateful）：
   - 因为存在对特定女性群体的偏见和贬低，属于仇恨言论，所以判断为“hate”。

从大城市回来的大龄剩女 | 不要找 | Sexism | hate [END]
所以你想表达什么⁉
1. 首先分析评论对象（Target）：
   - 文本中“你”明确指向一个对象，所以评论对象是“你”。
2. 接着看对应论点（Argument）：
   - 文本表达的是对“你”想要表达内容的疑问，并没有对“你”进行负面评价或攻击性描述，论点可理解为“不清楚想表达什么”。
3. 然后确定目标群体（Targeted Group）：
   - 此文本没有涉及到针对LGBTQ、Racism、Region、Sexism等仇恨相关的群体指向，所以目标群体是“non - hate”。
4. 最后判断是否仇恨（Hateful）：
   - 文本只是一个单纯的询问，没有仇恨、攻击性等负面情感，所以判断为“non - hate”。

你 | 不清楚想表达什么 | non - hate | non - hate [END]
才不对亢让t们团结鼓励让t们风况捏参考棒子锅
这段文本表述比较模糊，但从“棒子锅”的表述推测可能带有对韩国群体的不尊重。以下是分析：
1. **评论对象**：根据“棒子锅”推测评论对象可能是韩国人或与韩国相关事物。
2. **论点**：由于文本表述模糊，难以准确提炼出明确论点，不过整体语气可能带有负面态度。
3. **目标群体**：“棒子”是带有歧视性指代韩国人的词汇，所以目标群体属于Region（地区相关）。
4. **是否仇恨**：因为使用了歧视性词汇，所以属于仇恨言论。

韩国人或与韩国相关事物 | 因文本模糊难以明确但整体有负面态度 | Re




█████▊                                                                                                                                                                                           | 1/20 [00:05<01:44,  5.50s/it]







                                                                                                                                                                                  | 1/20 [00:05<01:46,  5.61s/it]

有一部分南方人是不是分不清北方口音啊 上次我在东莞看见两个人吵架 一个广东本地的 另一个明显是唐山口音 旁边就有人说:哪里的？听口音东北的吧😲
1. 首先分析评论对象（Target）：
   - 文本中提到“有一部分南方人”，所以评论对象是“有一部分南方人”。
2. 接着确定对应论点（Argument）：
   - 论点是“分不清北方口音”。
3. 然后判断目标群体（Targeted Group）：
   - 这里涉及到地域相关的评价，“南方人”是一个基于地域划分的群体，所以目标群体是“Region”。
4. 最后判断是否仇恨（Hateful）：
   - 只是在质疑一部分南方人分不清北方口音，没有明显的恶意攻击、贬低等仇恨性表达，所以是“non - hate”。

有一部分南方人 | 分不清北方口音 | Region | non - hate [END]
黑人，沐猩而冠。
1. 首先分析评论对象（Target）：
   - 从文本“黑人，沐猩而冠”可以明确看出，评论的对象就是“黑人”。
2. 接着提取对应论点（Argument）：
   - 文本中对黑人描述的内容是“沐猩而冠”，这就是表达对黑人的一种带有歧视性的观点，所以论点是“沐猩而冠”。
3. 然后确定目标群体（Targeted Group）：
   - “黑人”属于特定的种族群体，该言论是针对黑人这一种族带有侮辱性的表述，所以目标群体属于“Racism”（种族歧视相关）。
4. 最后判断是否仇恨（Hateful）：
   - “沐猩而冠”这样的表述是对黑人的贬低和侮辱，带有明显的仇恨和歧视意味，所以是“hate”。

黑人 | 沐猩而冠 | Racism | hate [END]
他是矮姿患者
1. 首先分析评论对象（Target）：
   - 从文本“他是矮姿患者”可以明确看出，评论的对象是“他”。
2. 接着提取对应论点（Argument）：
   - 文本中描述的论点就是“矮姿患者”。
3. 然后确定目标群体（Targeted Group）：
   - 这里并没有针对LGBTQ、Racism（种族）、Region（地域）、Sexism（性别）等特定仇恨群体的指向，只是对“他”的一种描述，所以目标群体为non - hate。
4. 最后判断是否仇恨（Hateful）：
   - 仅仅说“他是矮姿患者”，

...

黑龙江人还歧视辽宁吉林人，说他们的口音怎样怎样😄😄
1. 首先确定评论对象（Target）：
   - 从文本中可以看出，评论的对象是“黑龙江人”，他们针对的是“辽宁吉林人”。所以这里有两个评论对象，先看针对辽宁吉林人的部分，评论对象就是“辽宁吉林人”。
2. 接着提取对应论点（Argument）：
   - 关于辽宁吉林人的论点是“黑龙江人说他们的口音怎样怎样”，结合前面提到的“歧视”，这里可以总结论点为“被黑龙江人歧视口音”。
3. 然后确定目标群体（Targeted Group）：
   - 因为涉及到不同地区的人群（辽宁、吉林），所以目标群体属于“Region”（地域）。
4. 最后判断是否仇恨（Hateful）：
   - 文本中提到“歧视”，表明存在负面态度，所以属于“hate”（仇恨）。

辽宁吉林人 | 被黑龙江人歧视口音 | Region | hate [END]
看个人吧 有些欧洲小国是对中国人印象不错 但要谈婚论嫁还是得看个人能力
1. 首先分析评论对象（Target）：
   - 文本中提到“有些欧洲小国”和“中国人”，但整体是一种客观的讨论，没有针对特定的个体或群体进行攻击。“看个人吧”表明重点在于个人情况，所以评论对象这里更倾向于NULL。
2. 接着看对应论点（Argument）：
   - 文本表达“有些欧洲小国是对中国人印象不错，但要谈婚论嫁还是得看个人能力”，这是一种客观陈述观点，没有攻击性的言论，论点就是这句话所表达的客观内容。
3. 然后确定目标群体（Targeted Group）：
   - 文本没有对任何群体进行歧视或仇恨性的表达，只是客观描述不同群体之间的一种观点，所以目标群体是non - hate。
4. 最后判断是否仇恨（Hateful）：
   - 整个文本语气客观，没有仇恨、歧视等负面情感，所以是non - hate。

NULL | 有些欧洲小国是对中国人印象不错，但要谈婚论嫁还是得看个人能力 | non - hate | non - hate [END]









100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [02:12<00:00,  6.64s/it]

正常东北人是不歧视的，但不得不说，在东北，东北红脖子和东北奸都是有的，前者认为东北最好，目空一切；后者认为东北最差，南方是爹。
1. 首先分析评论对象（Target）：
   - 从文本中可以看到，作者提到了“东北红脖子”和“东北奸”，所以评论对象就是“东北红脖子”和“东北奸”。
2. 接着提取对应论点（Argument）：
   - 对于“东北红脖子”，论点是“认为东北最好，目空一切”；对于“东北奸”，论点是“认为东北最差，南方是爹”。
3. 然后确定目标群体（Targeted Group）：
   - 这里“东北红脖子”和“东北奸”都是带有地域指向的负面称呼，针对的是特定地区（东北）的人群，所以目标群体属于“Region”（地域）。
4. 最后判断是否仇恨（Hateful）：
   - 作者使用了带有侮辱性的词汇“东北红脖子”和“东北奸”来描述部分东北人，这种表述带有仇恨情绪，所以判断为“hate”。

东北红脖子 | 认为东北最好，目空一切 | Region | hate [SEP] 东北奸 | 认为东北最差，南方是爹 | Region | hate [END]









100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [02:16<00:00,  6.83s/it]

先解决孩子安全问题再说吧，基佬炼铜是人尽皆知的
1. 首先分析评论对象（Target）：从文本“基佬炼铜是人皆知皆知的”可以看出，评论对象是“基佬”。
2. 接着提取对应论点（Argument）：论点是“炼铜”，在网络语境中“炼铜”指恋童这种违背公序良俗和法律的行为，这里是对“基佬”群体恶意的不实指控。
3. 然后确定目标群体（Targeted Group）：“基佬”通常是对男同性恋者带有歧视性的称呼，所以目标群体属于LGBTQ。
4. 最后判断是否仇恨（Hateful）：将恋童这种严重负面行为无端加诸于LGBTQ群体中的男同性恋者，是对该群体的恶意诋毁和攻击，属于仇恨言论。

基佬 | 炼铜 | LGBTQ | hate [END]


In [7]:
final_res_last_line = []
for s in final_res:
    # 使用 splitlines() 方法按行分割字符串
    lines = s.splitlines()
    
    # 取最后一行
    last_line = lines[-1]
    last_line = last_line.replace('non - hate','non-hate')
    final_res_last_line.append(last_line)

In [8]:
final_res_last_line

['飞周兄弟 | 不认亲、还袭击 | others | non-hate [END]',
 '广黑吧 | 询问是否存在 | Region | hate [END]',
 '卫辉官媒 | 错误 | Region | non-hate [SEP] 你 | 可能在造谣 | others | non-hate [END]',
 '男权 | 不应该被污名化，男权是男女平权 | non-hate | non-hate [END]',
 '输出： 黑人 | 负面态度（通过歧视性词汇体现） | Racism | hate [END]',
 '你 | 追求男女平等 | non-hate | non-hate [END]',
 '最终输出：你 | 别以为中国人会吃政治正确那一套 | non-hate | non-hate [END]',
 '中国人 | 搞不懂为啥要吹黑人性能力 | others | non-hate [SEP] 黑人 | 相关黑人性能力的谣言 | Racism | hate [END]',
 '田楠 | 骗婚骗炮，水准不够导致有剩女 | others | hate [END]',
 '输出：这个视频 | 比较恶臭 | non-hate | non-hate [END]',
 '这位答主 | 按你的说法答主也该滚出中国 | others | hate [END]',
 '异性恋 | 理解＋不接受 | others | hate [END]',
 '输出：NULL | 真的要检讨一下，广东常常刮台风，下大雨，最多也只是水浸街 | non-hate | non-hate [END]',
 '输出：有人 | 为了骂原，甚至会和女拳站一边 | Sexism | hate [END]',
 '法国 | 完蛋了 | Region | hate [END]',
 '带头的 | 恰烂钱 | others | hate [SEP] 跟风的 | 铁nt | others | hate [END]',
 '她 | 跨种族跨文化优越感就会变得越充实，就会越相信黑人的社会文化。这简直是养虎为患 | Racism | hate [END]',
 '输出：阿三（指代印度人） | 未来可能会像阿三一样四处黑公交 | Racism | hate [SEP] 日本人民 | 日本低欲望是因为自身人民变得太懒

In [9]:
# 定义要保存的文件路径
file_path = f'../dataset/res/doubao-{mode}-think0303.txt'

# 打开文件以写入模式
with open(file_path, 'w', encoding='utf-8') as file:
    # 遍历列表中的每个元素
    for line in final_res_last_line:
        # 将元素写入文件，并添加换行符
        file.write(line + '\n')

print(f"数据已成功保存到 {file_path}")

数据已成功保存到 ../dataset/res/doubao-chat-think0303.txt
